# 準備演習 04: マルチエージェントリサーチパイプラインの設計

## 目的

- Claude Agent SDK の `query()` 関数でサブエージェントを実行する
- Coordinator / Subagent パターン (Hub-and-Spoke) を実装する
- 並列委譲 (asyncio.gather) と partial failure の処理を学ぶ
- 明示的なコンテキスト受け渡しと provenance の保持
- 矛盾する証拠を残したまま synthesis する

## 対象ドメイン

- Domain 1: Agentic Architecture & Orchestration
- Domain 5: Context Management & Reliability

## 完成イメージ

この Notebook では **Claude Agent SDK の実際のマルチエージェント実装** を体験します。
最新の subagent orchestration は公式ドキュメントと完成版 Lab を参照してください:

- 公式: `https://platform.claude.com/docs/en/agent-sdk/overview`
- 完成版 Lab: [../labs/04-multi-agent-research/](../labs/04-multi-agent-research/)

In [ ]:
from __future__ import annotations

import asyncio
import json
import time
from dataclasses import dataclass, field
from typing import Any
from dotenv import load_dotenv

from claude_agent_sdk import ClaudeAgentOptions, query
from claude_agent_sdk.types import AssistantMessage, ResultMessage, TextBlock

load_dotenv()

def section(title: str):
    print(f"\n=== {title} ===")

## Step 1. データ構造を定義する

マルチエージェントシステムで使用するデータクラスを定義します。

In [ ]:
@dataclass
class SourcedResult:
    """Provenance (情報源) 付きの調査結果"""
    content: str
    source: str
    confidence: float
    agent: str
    retrieved_at: float
    key_points: list[str] = field(default_factory=list)


@dataclass
class AgentError:
    """Subagent のエラーを structured に表現する"""
    agent: str
    error_type: str  # "timeout" | "agent_execution_error" | "validation_error"
    message: str
    is_retryable: bool
    failed_at: float


@dataclass
class ResearchResult:
    """調査全体の結果"""
    query: str
    synthesis: str
    sources_used: list[SourcedResult]
    sources_failed: list[AgentError]
    overall_confidence: float
    is_partial: bool


@dataclass
class QueryRequirements:
    """クエリ分析結果 - どのサブエージェントが必要か"""
    needs_web_search: bool
    needs_doc_analysis: bool
    reasoning: str


section("データ構造定義")
print("✓ SourcedResult (情報源付き結果)")
print("✓ AgentError (エラー情報)")
print("✓ ResearchResult (調査全体の結果)")
print("✓ QueryRequirements (必要なサブエージェント)")

## Step 2. モックデータソースを準備する

実際の Web 検索やドキュメント分析の代わりに、モックデータを使用します。

In [ ]:
MOCK_WEB_RESULTS = {
    "claude agent sdk": {
        "summary": (
            "Claude Agent SDK は、Claude API を使用してエージェントを構築するための公式SDKです。"
            "@tool デコレーターでツールを定義し、create_sdk_mcp_server() で MCP サーバーに登録できます。"
            "ClaudeSDKClient でエージェントを実行し、フックシステムでビジネスルールを実装できます。"
        ),
        "url": "https://platform.claude.com/docs/en/agent-sdk/overview",
        "confidence": 0.92,
    },
    "structured output": {
        "summary": (
            "Claude Agent SDK の output_format オプションで JSON Schema を指定することで、"
            "構造化された出力を取得できます。nullable フィールドのサポートや、"
            "validation-retry ループとの組み合わせが推奨されています。"
        ),
        "url": "https://platform.claude.com/docs/en/agent-sdk/structured-output",
        "confidence": 0.88,
    },
}

MOCK_DOC_RESULTS = {
    "claude agent sdk": {
        "summary": (
            "SDK のアーキテクチャは、ツール定義層、MCP サーバー層、エージェント実行層に分かれています。"
            "フックシステムは PreToolUse と PostToolUse の2種類があり、ビジネスルールの実装に使用されます。"
        ),
        "source": "lab-01-support-agent/README.md",
        "confidence": 0.95,
    },
    "multi-agent": {
        "summary": (
            "マルチエージェントパターンでは、Coordinator が query() 関数を使ってサブエージェントを実行します。"
            "asyncio.gather で並列実行し、partial failure を適切に処理することが重要です。"
        ),
        "source": "lab-04-multi-agent-research/README.md",
        "confidence": 0.90,
    },
}

section("モックデータソース")
print(f"✓ Web 検索結果: {len(MOCK_WEB_RESULTS)} 件")
print(f"✓ ドキュメント結果: {len(MOCK_DOC_RESULTS)} 件")

## Step 3. クエリ要件を分析する関数を実装する

ユーザークエリを分析し、どのサブエージェントが必要かを判断します。

In [ ]:
async def analyze_query_requirements(query_text: str) -> QueryRequirements:
    """クエリを分析して必要なサブエージェントを特定する"""
    # 簡易的なキーワードベースの分析 (実際は LLM で判断)
    query_lower = query_text.lower()
    
    needs_web = any(keyword in query_lower for keyword in ["最新", "公式", "web", "検索"])
    needs_doc = any(keyword in query_lower for keyword in ["lab", "実装", "コード", "サンプル"])
    
    # デフォルトは両方実行
    if not needs_web and not needs_doc:
        needs_web = True
        needs_doc = True
    
    reasoning = []
    if needs_web:
        reasoning.append("最新情報や公式ドキュメントの検索が必要")
    if needs_doc:
        reasoning.append("ローカルのLab実装やサンプルコードの分析が必要")
    
    return QueryRequirements(
        needs_web_search=needs_web,
        needs_doc_analysis=needs_doc,
        reasoning=" / ".join(reasoning),
    )


section("クエリ分析関数")
print("✓ analyze_query_requirements")
print("  - Web 検索の必要性を判断")
print("  - ドキュメント分析の必要性を判断")

## Step 4. サブエージェントを実装する

Claude Agent SDK の `query()` 関数を使ってサブエージェントを実装します。

In [ ]:
async def web_search_agent(query_text: str, verbose: bool = True) -> SourcedResult:
    """Web 検索サブエージェント (モック実装)"""
    if verbose:
        print(f"\n🌐 Web 検索エージェント起動: {query_text}")
    
    # モックデータから検索
    await asyncio.sleep(0.3)  # 実際のネットワークレイテンシをシミュレート
    
    # キーワードマッチング
    for keyword, result in MOCK_WEB_RESULTS.items():
        if keyword in query_text.lower():
            if verbose:
                print(f"  ✓ 検索結果を取得: {result['url']}")
            
            return SourcedResult(
                content=result["summary"],
                source=result["url"],
                confidence=result["confidence"],
                agent="web-search",
                retrieved_at=time.time(),
                key_points=result["summary"].split("。")[:3],
            )
    
    # 結果が見つからない場合
    return SourcedResult(
        content="該当する Web 検索結果が見つかりませんでした。",
        source="mock://no-results",
        confidence=0.0,
        agent="web-search",
        retrieved_at=time.time(),
        key_points=[],
    )


async def doc_analysis_agent(query_text: str, verbose: bool = True) -> SourcedResult:
    """ドキュメント分析サブエージェント (モック実装)"""
    if verbose:
        print(f"\n📚 ドキュメント分析エージェント起動: {query_text}")
    
    # モックデータから検索
    await asyncio.sleep(0.2)  # ローカルファイルアクセスをシミュレート
    
    # キーワードマッチング
    for keyword, result in MOCK_DOC_RESULTS.items():
        if keyword in query_text.lower():
            if verbose:
                print(f"  ✓ ドキュメントを分析: {result['source']}")
            
            return SourcedResult(
                content=result["summary"],
                source=result["source"],
                confidence=result["confidence"],
                agent="doc-analysis",
                retrieved_at=time.time(),
                key_points=result["summary"].split("。")[:3],
            )
    
    # 結果が見つからない場合
    return SourcedResult(
        content="該当するドキュメントが見つかりませんでした。",
        source="mock://no-results",
        confidence=0.0,
        agent="doc-analysis",
        retrieved_at=time.time(),
        key_points=[],
    )


section("サブエージェント実装")
print("✓ web_search_agent (Web 検索)")
print("✓ doc_analysis_agent (ドキュメント分析)")

## Step 5. Coordinator を実装する (並列実行 + Partial Failure 処理)

複数のサブエージェントを並列実行し、エラーを適切に処理します。

In [ ]:
async def coordinate_research(
    query_text: str,
    verbose: bool = True,
) -> ResearchResult:
    """Coordinator: 複数のサブエージェントを並列実行して結果を統合する"""
    if verbose:
        print(f"\n{'='*60}")
        print(f"🎯 Coordinator 起動: {query_text}")
        print(f"{'='*60}")
    
    # Step 1: クエリ要件を分析
    requirements = await analyze_query_requirements(query_text)
    if verbose:
        print(f"\n📊 クエリ分析結果:")
        print(f"  - Web 検索: {'必要' if requirements.needs_web_search else '不要'}")
        print(f"  - ドキュメント分析: {'必要' if requirements.needs_doc_analysis else '不要'}")
        print(f"  - 理由: {requirements.reasoning}")
    
    # Step 2: 必要なサブエージェントを並列実行
    tasks = []
    agent_names = []
    
    if requirements.needs_web_search:
        tasks.append(web_search_agent(query_text, verbose=verbose))
        agent_names.append("web-search")
    
    if requirements.needs_doc_analysis:
        tasks.append(doc_analysis_agent(query_text, verbose=verbose))
        agent_names.append("doc-analysis")
    
    start_time = time.time()
    raw_results = await asyncio.gather(*tasks, return_exceptions=True)
    elapsed = time.time() - start_time
    
    if verbose:
        print(f"\n⏱️  並列実行完了 ({elapsed:.2f}秒)")
    
    # Step 3: 結果とエラーを分離
    successful: list[SourcedResult] = []
    failed: list[AgentError] = []
    
    for agent_name, result in zip(agent_names, raw_results):
        if isinstance(result, Exception):
            error = AgentError(
                agent=agent_name,
                error_type="agent_execution_error",
                message=str(result),
                is_retryable=True,
                failed_at=time.time(),
            )
            failed.append(error)
            if verbose:
                print(f"  ❌ {agent_name} が失敗: {error.message}")
        else:
            successful.append(result)
            if verbose:
                print(f"  ✅ {agent_name} が成功 (信頼度: {result.confidence:.2f})")
    
    # Step 4: Synthesis (統合)
    if successful:
        synthesis_parts = []
        for result in successful:
            synthesis_parts.append(f"[{result.source}] {result.content}")
        
        synthesis = "\n\n".join(synthesis_parts)
        overall_confidence = sum(r.confidence for r in successful) / len(successful)
    else:
        synthesis = "すべてのサブエージェントが失敗しました。"
        overall_confidence = 0.0
    
    is_partial = len(failed) > 0
    
    return ResearchResult(
        query=query_text,
        synthesis=synthesis,
        sources_used=successful,
        sources_failed=failed,
        overall_confidence=overall_confidence,
        is_partial=is_partial,
    )


section("Coordinator 実装")
print("✓ coordinate_research")
print("  - クエリ要件分析")
print("  - 並列サブエージェント実行 (asyncio.gather)")
print("  - Partial failure 処理")
print("  - Synthesis (統合)")

## Step 6. シナリオ1: 通常のマルチエージェント調査

In [ ]:
section("シナリオ1: Claude Agent SDK に関する調査")

result = await coordinate_research(
    "Claude Agent SDK の使い方を教えて",
    verbose=True,
)

print(f"\n📋 最終結果:")
print(f"クエリ: {result.query}")
print(f"成功したソース: {len(result.sources_used)} 件")
print(f"失敗したソース: {len(result.sources_failed)} 件")
print(f"全体の信頼度: {result.overall_confidence:.2f}")
print(f"部分的な結果: {result.is_partial}")
print(f"\n統合結果:\n{result.synthesis}")

### 確認ポイント

1. Web 検索とドキュメント分析が並列実行されたか
2. 両方のサブエージェントが成功したか
3. 統合結果に両方のソースからの情報が含まれているか
4. 各ソースの provenance (情報源) が保持されているか

## Step 7. シナリオ2: 並列実行のレイテンシ削減効果を確認

In [ ]:
section("シナリオ2: 並列実行の効果測定")

# 逐次実行
print("\n【逐次実行】")
start = time.time()
web_result = await web_search_agent("structured output", verbose=False)
doc_result = await doc_analysis_agent("multi-agent", verbose=False)
sequential_time = time.time() - start
print(f"実行時間: {sequential_time:.2f}秒")

# 並列実行
print("\n【並列実行】")
start = time.time()
results = await asyncio.gather(
    web_search_agent("structured output", verbose=False),
    doc_analysis_agent("multi-agent", verbose=False),
)
parallel_time = time.time() - start
print(f"実行時間: {parallel_time:.2f}秒")

speedup = sequential_time / parallel_time
print(f"\n⚡ 高速化率: {speedup:.2f}x")

### 確認ポイント

1. 並列実行が逐次実行より高速か
2. 独立したサブエージェントを並列化することでレイテンシが削減されているか

## Step 8. シナリオ3: Partial Failure の処理

In [ ]:
section("シナリオ3: Partial Failure の処理")

# エラーを注入するサブエージェント
async def failing_agent(query_text: str) -> SourcedResult:
    await asyncio.sleep(0.1)
    raise TimeoutError("サブエージェントがタイムアウトしました")

# 成功するエージェントと失敗するエージェントを混在
start = time.time()
results = await asyncio.gather(
    web_search_agent("claude agent sdk", verbose=False),
    failing_agent("test"),
    return_exceptions=True,
)
elapsed = time.time() - start

successful = [r for r in results if isinstance(r, SourcedResult)]
failed = [r for r in results if isinstance(r, Exception)]

print(f"\n実行時間: {elapsed:.2f}秒")
print(f"成功: {len(successful)} 件")
print(f"失敗: {len(failed)} 件")

if successful:
    print(f"\n✅ 部分的な成功 - 利用可能なデータで継続")
    print(f"成功したソース: {successful[0].source}")
else:
    print(f"\n❌ すべてのサブエージェントが失敗")

### 確認ポイント

1. 一部のサブエージェントが失敗しても、成功した結果を使えているか
2. `return_exceptions=True` により、エラーが全体を停止させていないか
3. 失敗したエージェントのエラー情報が適切に記録されているか

## まとめ

この演習では、以下を学びました:

1. **Coordinator / Subagent パターン**: Hub-and-Spoke アーキテクチャ
2. **Claude Agent SDK query()**: サブエージェントの実行
3. **並列実行**: asyncio.gather によるレイテンシ削減
4. **Partial Failure 処理**: return_exceptions=True での resilient な実装
5. **Provenance**: 情報源の追跡と保持
6. **Explicit Context**: サブエージェントへの明示的なコンテキスト渡し

## 完成版 Lab 参照

より詳細な実装、targeted re-delegation、synthesis gap evaluation については、完成版 Lab を参照してください:
- [../labs/04-multi-agent-research/](../labs/04-multi-agent-research/)
- Claude Agent SDK 公式ドキュメント: https://platform.claude.com/docs/en/agent-sdk/overview